In [1]:
import pandas as pd
import panel as pn
import matplotlib.pyplot as plt
import seaborn as sns

pn.extension()
sns.set_theme(style="whitegrid")

In [2]:
df = pd.read_csv("credit_risk_dataset.csv")
df = df[df['person_age'] <= 100]

In [3]:
df['loan_int_rate'] = df['loan_int_rate'].fillna(df['loan_int_rate'].median())

df['person_emp_length'] = df['person_emp_length'].fillna(df['person_emp_length'].median())

df['person_income'] = df['person_income'].clip(upper=df['person_income'].quantile(0.99))

df['loan_amnt'] = df['loan_amnt'].clip(upper=df['loan_amnt'].quantile(0.99))

df['loan_percent_income'] = df['loan_percent_income'].clip(upper=df['loan_percent_income'].quantile(0.99))

df['Status_Label'] = df['loan_status'].map({0: 'Healthy', 1: 'Default'})

df['income_lakh'] = df['person_income'] / 100000


In [4]:
df_german = pd.read_csv("german_credit_data.csv")
df_german['Default'] = (df_german['target'] == 'bad').astype(int)
df_german['sex'] = df_german['status_and_sex'].apply(lambda x: x.split(' : ')[0] if isinstance(x, str) else str(x))

In [5]:
age_slider = pn.widgets.IntRangeSlider(
    name="Age Range (Universal)",
    start=min(int(df['person_age'].min()), int(df_german['age'].min())),
    end=max(int(df['person_age'].max()), int(df_german['age'].max())),
    value=(20, 60)
)

In [6]:
intent_filter = pn.widgets.MultiChoice(
    name="Loan Intent (DS1)",
    options=list(df['loan_intent'].unique()),
    value=list(df['loan_intent'].unique())
)

grade_filter = pn.widgets.MultiChoice(
    name="Loan Grade (DS1)",
    options=sorted(df['loan_grade'].unique()),
    value=sorted(df['loan_grade'].unique())
)

status_filter = pn.widgets.MultiChoice(
    name="Loan Status (DS1)",
    options=['Healthy', 'Default'],
    value=['Healthy', 'Default']
)

In [7]:
dashboard_output = pn.Column(sizing_mode="stretch_width")

In [8]:
def build_kpi(d):
    return pn.Row(
        pn.indicators.Number(name="Total Loans", value=len(d), font_size='18pt', title_size='14pt'),
        pn.indicators.Number(name="Total Exposure", value=int(d['loan_amnt'].sum()), font_size='18pt', title_size='14pt'),
        pn.indicators.Number(name="Default Rate", value=(d['loan_status'].mean()*100 if not d.empty else 0), format="{value:.1f}%", font_size='18pt', title_size='14pt'),
        pn.indicators.Number(name="Avg Interest", value=(d['loan_int_rate'].mean() if not d.empty else 0), format="{value:.2f}%", font_size='18pt', title_size='14pt'),
        sizing_mode="stretch_width"
    )

In [9]:
def build_ds1_plots(d):
    fig1, ax1 = plt.subplots(figsize=(6, 4))
    if not d.empty:
        sns.violinplot(data=d, x='person_home_ownership', y='income_lakh', ax=ax1, color="#3b82f6")
    ax1.set_title("Income vs Home Ownership", fontweight='bold')
    ax1.set_xlabel("")
    plt.close(fig1)

    fig2, ax2 = plt.subplots(figsize=(6, 4))
    if not d.empty:
        g = d.groupby('loan_grade')['loan_status'].mean().reset_index()
        g['Default %'] = g['loan_status'] * 100
        sns.barplot(data=g, x='loan_grade', y='Default %', ax=ax2, color="#3b82f6")
    ax2.set_title("Default % by Credit Grade", fontweight='bold')
    ax2.set_xlabel("")
    plt.close(fig2)

    return fig1, fig2

In [10]:
def build_risk_plots(d):
    fig3, ax3 = plt.subplots(figsize=(6, 4))
    if not d.empty:
        sns.boxplot(data=d, x='Status_Label', y='loan_percent_income', hue='Status_Label', ax=ax3, palette=['#3b82f6', '#ef4444'], legend=False)
    ax3.set_title("Loan Burden vs Default", fontweight='bold')
    ax3.set_xlabel("")
    plt.close(fig3)

    fig4, ax4 = plt.subplots(figsize=(6, 4))
    if not d.empty:
        hm = d.groupby(['loan_intent', 'loan_grade'])['loan_status'].mean().reset_index()
        hm['Default %'] = hm['loan_status'] * 100
        pivot = hm.pivot(index='loan_intent', columns='loan_grade', values='Default %')
        sns.heatmap(pivot, cmap="Reds", ax=ax4)
    ax4.set_title("Risk Heatmap", fontweight='bold')
    ax4.set_xlabel("")
    ax4.set_ylabel("")
    plt.close(fig4)

    return fig3, fig4

In [11]:
def build_german_plots(d2):
    fig5, ax5 = plt.subplots(figsize=(10, 4))
    if not d2.empty:
        g2 = d2.groupby('sex')['Default'].mean().reset_index()
        g2['Default %'] = g2['Default'] * 100
        sns.barplot(data=g2, x='sex', y='Default %', ax=ax5, color="#8b5cf6")
    ax5.set_title("Default % by Gender (German Data)", fontweight='bold')
    ax5.set_xlabel("")
    plt.close(fig5)

    fig6, ax6 = plt.subplots(figsize=(10, 4))
    if not d2.empty:
        sns.scatterplot(data=d2, x='month_duration', y='credit_amount', hue='target', alpha=0.7, palette=['#10b981', '#ef4444'], ax=ax6)
    ax6.set_title("Duration vs Credit Amount (German Data)", fontweight='bold')
    plt.close(fig6)

    return fig5, fig6

In [12]:
def update_dashboard(event=None):
    d = df[
        (df['person_age'] >= age_slider.value[0]) &
        (df['person_age'] <= age_slider.value[1]) &
        (df['loan_intent'].isin(intent_filter.value)) &
        (df['loan_grade'].isin(grade_filter.value)) &
        (df['Status_Label'].isin(status_filter.value))
    ]

    d2 = df_german[
        (df_german['age'] >= age_slider.value[0]) &
        (df_german['age'] <= age_slider.value[1])
    ]

    dashboard_output.clear()

    kpi = build_kpi(d)
    fig1, fig2 = build_ds1_plots(d)
    fig3, fig4 = build_risk_plots(d)
    fig5, fig6 = build_german_plots(d2)

    dashboard_output.append(pn.pane.Markdown("## Key Metrics"))
    dashboard_output.append(kpi)
    dashboard_output.append(pn.layout.Divider())

    dashboard_output.append(pn.pane.Markdown("## Income & Credit Analysis (Dataset 1)"))
    dashboard_output.append(
        pn.Row(
            pn.pane.Matplotlib(fig1, tight=True, sizing_mode="scale_width"),
            pn.pane.Matplotlib(fig2, tight=True, sizing_mode="scale_width"),
            sizing_mode="stretch_width"
        )
    )

    dashboard_output.append(pn.pane.Markdown("## Risk & Burden Analysis (Dataset 1)"))
    dashboard_output.append(
        pn.Row(
            pn.pane.Matplotlib(fig3, tight=True, sizing_mode="scale_width"),
            pn.pane.Matplotlib(fig4, tight=True, sizing_mode="scale_width"),
            sizing_mode="stretch_width"
        )
    )

    dashboard_output.append(pn.layout.Divider())
    dashboard_output.append(pn.pane.Markdown("## Demographic Risk Profiling (German Dataset)"))
    dashboard_output.append(pn.pane.Matplotlib(fig5, tight=True, sizing_mode="stretch_width"))

    dashboard_output.append(pn.pane.Markdown("## Loan Duration & Amount (German Dataset)"))
    dashboard_output.append(pn.pane.Matplotlib(fig6, tight=True, sizing_mode="stretch_width"))

In [13]:
age_slider.param.watch(update_dashboard, 'value')
intent_filter.param.watch(update_dashboard, 'value')
grade_filter.param.watch(update_dashboard, 'value')
status_filter.param.watch(update_dashboard, 'value')

update_dashboard()

In [14]:
template = pn.template.BootstrapTemplate(
    title="Dual-Data Credit Risk Dashboard",
    sidebar=[
        pn.pane.Markdown("## Filters"),
        age_slider,
        intent_filter,
        grade_filter,
        status_filter
    ]
)

template.main.append(pn.pane.Markdown("# Dual-Data Credit Risk Intelligence Dashboard", styles={'font-size': '28px', 'font-weight': 'bold'}))
template.main.append(pn.pane.Markdown("### Analyze borrower risk and financial behavior across two independent datasets.", styles={'color': '#555'}))
template.main.append(dashboard_output)

template.show()

Launching server at http://localhost:55140
